Suppose we have a single outcome. Then for each time point, our data consists of:
$$
y_{1t},\quad \vec{Y}_t = (y_{2t}, \ldots y_{Nt}),
$$
the outcome for the treated unit and the outcomes for the control units, respectively.

The objective is to learn a matrix $Q$ such that:
$$
\sum_{t=1}^{T_0} \bigg(y_{1t} - \vec{Y}_t\mathbf{Q}\vec{w}\bigg)^2 
$$
is minimized.
The dimension of $\mathbf{Q}$ is $\mathbb{R}^{N-1, D}$ where $D$ is a chosen embedding dimension.
The data are generated according to:
$$
[\vec{y}_{1t} \mathbf{Y}_t] = (\Lambda \mathbf{f}_t)^T
$$
where 
$$
\mathbf{\Lambda} \in \mathbb{R}^{N\times f},\quad\mathbf{f}_t \in \mathbb{R}^{f\times m},
$$
with $m=1$. Since $m=1$, the factor matrix $\mathbf{f}_t$ is just a vector $\vec{f}_t\in \mathbb{R}^{f\times 1}$. In addition,
$$
\vec{Y}_t = (\tilde{\Lambda} \vec{f}_t)^T = \vec{f}_t^T \tilde{\Lambda}^T,
$$
where $\tilde{\Lambda}$ contains only the loadings for the control units.

Observe that 
$$
y_{1t} \approx \vec{Y}_t \mathbf{Q}\vec{w} \implies \vec{f}_t \lambda_1 = \vec{f}_t^T \tilde{\Lambda}^T \mathbf{Q}\vec{w} \implies \lambda_1 = \tilde{\Lambda}^T \mathbf{Q}\vec{w},
$$ 
where the first implication follows by applying the DGP, and the second by cancellation.

Further, if $\mathbf{Q} = \tilde{\Lambda}(\tilde{\Lambda}^T\tilde{\Lambda})^{-1}$, then 
$$
\lambda_1 = I_f \vec{w} \implies \vec{w} = \lambda_1,
$$ 
which is problematic, since in the SCM, the weights are restricted to be positive. However, if there was no restriction on the weights, $\mathbf{Q} = \tilde{\Lambda}(\tilde{\Lambda}^T\tilde{\Lambda})^{-1}$ would be optimal. In this case, $\mathbf{Q}$ would recover the factor loadings for the treated unit through the weights $\vec{w}$. Perhaps relaxing the simplex constraint on $\vec{w}$ yields this kind of behavior. The only issue is that the solution is not unique since $\mathbf{Q}M M^{-1}\vec{w}$ gives the same solution.

In [41]:
import numpy as np
import torch
import learnQorthogonal as lq
import learnQ as lqreg

torch.manual_seed(42)
np.random.seed(42)

def simple_sim_Data(T,m,f,N):
    Outcomes = []
    
    # Fixed factor loadings
    Lambda = torch.randn(N, f, dtype=torch.float64)
    # Initial factors
    F = torch.randn(f, m, dtype=torch.float64)

    # just add noise to the original factors at each step. No AR process here.
    for t in range(T):
        noise = torch.randn(f, m, dtype=torch.float64) * 0.01
        F = F + noise
        Y_t = (Lambda @ F).T
        # print("Y_t shape:", Y_t.shape)
        Outcomes.append(Y_t)

    return Outcomes, Lambda


In [44]:
# Generate synthetic data          T, m, f, N
outcomes, Lambda = simple_sim_Data(10, 1, 6, 5)

print("Lambda: ",Lambda)
print("Shape of Lambda:",Lambda.shape)
print("Loadings for unit 1:", Lambda[0])
t_Lambda = Lambda[1:,:]
print("t_Lambda:",t_Lambda)
print("Shape of t_Lambda:",t_Lambda.shape)

# Separate the simulated data into targets and predictors
targets = []
predictors = []
for outcome in outcomes:
    # take the first column
    targets.append(outcome[:, 0])
    # all but the first column
    predictors.append(outcome[:, 1:])

final_target = targets.pop() 
final_predictors = predictors.pop()

# You can inspect the data structure by uncommenting these lines
### --------------------------------------------------------------------- ###
# print("All outcomes: \n")
# for o in outcomes:
#     print(o)
#     print("\n")

# print("Targets: \n")
# for t in targets:
#     print(t,"\n")

# print("Predictors: \n")
# for p in predictors:
#     print(p,"\n")

# print("Final target: \n", final_target)
# print("Final predictors: \n", final_predictors)
### --------------------------------------------------------------------- ###


Lambda:  tensor([[ 0.38, -0.11, -1.46,  0.63,  0.00, -0.43],
        [ 0.97, -1.00, -0.62, -1.40, -0.60,  0.29],
        [-0.31, -1.09, -0.09,  0.90,  1.45,  1.36],
        [ 0.25, -0.95,  0.81, -0.44,  0.52, -2.05],
        [-0.12,  0.38, -0.79,  1.65, -1.28,  1.18]], dtype=torch.float64)
Shape of Lambda: torch.Size([5, 6])
Loadings for unit 1: tensor([ 0.38, -0.11, -1.46,  0.63,  0.00, -0.43], dtype=torch.float64)
t_Lambda: tensor([[ 0.97, -1.00, -0.62, -1.40, -0.60,  0.29],
        [-0.31, -1.09, -0.09,  0.90,  1.45,  1.36],
        [ 0.25, -0.95,  0.81, -0.44,  0.52, -2.05],
        [-0.12,  0.38, -0.79,  1.65, -1.28,  1.18]], dtype=torch.float64)
Shape of t_Lambda: torch.Size([4, 6])


In [45]:
# Fit the model
Q_final, w_final = lq.learnQorthogonal(
    targets,
    predictors,
    embedding_dim = 6,
    n_iterations=2000,
    reg_Q=0,
    reg_w=0,
    verbose=False,
    ### Un-comment to use the orthogonal version
    num_timepoints=None,
    init_Q="eye",
    fixed_weights=False
)

print("The learned Q matrix: \n",Q_final)
print("The final weights are:", w_final)
print("The factor loadings for the first unit are: ", Lambda[0])

# a better? product of the factors loadings
t_omega = torch.linalg.inv(t_Lambda.T @ t_Lambda) @ t_Lambda.T
t_omega = t_omega.T
#print("\nThe product of the factor loadings:\n", omega)
#print("The shape of omega:", omega.shape)


# can uncomment - algorithm produces good estimates
### ----------------------------------------
# print("\nFinal Target:\n", final_target)
# prediction = final_predictors @ Q_final @ w_final
# print("\nFinal prediction:\n", prediction)

# normalize by the first column, and then drop it from the matrix
# omega = omega / omega[:,[0]]
# omega = (omega[:,1:]).T

print("t_omega:",t_omega)

The learned Q matrix: 
 [[ 0.15528542 -0.92148458  0.18972523  0.15492864 -0.18269375 -0.18269386]
 [-0.34565844  0.17725817 -0.101485    0.80897099 -0.30361742 -0.30361651]
 [-0.56255716  0.00996921  0.13142358 -0.54739866 -0.42808442 -0.42808449]
 [ 0.34095497 -0.03938648 -0.85120781 -0.14801752 -0.26051349 -0.26051337]]
The final weights are: [6.05364758e-03 8.55715708e-04 9.20590523e-01 7.24989756e-02
 5.68920324e-07 5.68919959e-07]
The factor loadings for the first unit are:  tensor([ 0.38, -0.11, -1.46,  0.63,  0.00, -0.43], dtype=torch.float64)
t_omega: tensor([[ 3.42, -0.93,  3.19,  1.79, -0.15,  0.95],
        [-2.79, -0.85, -2.03, -0.69, -0.28, -2.26],
        [-1.99, -0.14, -2.03, -3.23, -0.07, -0.63],
        [-1.14, -1.89, -0.85,  3.55, -0.63, -1.19]], dtype=torch.float64)
